# KAVI resource cleaning

Run the code cells from top to bottom. The notebook reads these files from the repository's `data/` folder:

- `NYPL Connections Resources.csv`
- `NYPLConnections_NoMatch.csv`

It writes two files to `data/cleaned/`:

- `resources_clean.csv` - the complete cleaned resource data, including empty `latitude` and `longitude` fields for future geocoding
- `review_list.csv` - rows that need manual review by a human

The notebook:

1. Loads all CSV values as text and trims whitespace.
2. Handles unnamed rows by copying address details or adding extra text to the preceding row.
3. Checks whether each resource also appears in the NoMatch file and records that in `in_nomatch_file`. (is derived from original provided .csv files as a note but no geo-data exported yet, therefore not relevant; see 4.)
4. Leaves `latitude` and `longitude` empty; coordinates are not imported from ArcGIS or the NoMatch file.
5. Cleans names, addresses, ZIP codes, and website URLs.
6. Derives boroughs, row types, service types, review flags, and resource IDs.
7. Exports the cleaned data and review list.

Rows with uncertain classifications or source notes are marked with `needs_review` and `review_notes`.

## How to clean the review list (manual check for later, not for MVP relevant)

`review_list.csv` is a generated work list, not the source of truth. Review each row, then update the cleaning rules or source data and run the notebook again. Do not rely on edits made only in `review_list.csv`, because the next run will overwrite it.

Use `review_notes` to decide what to check:

- `service_type to assign`: set or extend the service-type mapping in the enrichment cell.
- `row_type needs a decision`: update the row-type logic or add an explicit exception.
- `Same place under several chapters`: check whether the duplicate locations are valid separate listings.
- `Original note:` or `Address-only row`: verify the source row and adjust the source-cleaning logic if needed.
- `In NoMatch file`: this is provenance only; it does not mean that coordinates should be imported.

After each review pass, run all cells from the beginning. Confirm that `needs_review` decreases and that the corrected values appear in `resources_clean.csv`.

## Accuracy safeguards

The notebook applies one service type to all rows with the same normalized organization name only when exactly one safe type is known. If an organization has conflicting types, `service_type` is cleared for all its rows and every row is flagged in `needs_review`; no unsafe choice is made. The main `resources_clean.csv` contains every row, including these flagged rows. `review_list.csv` is only a filtered convenience view. The audit fields `service_type_source`, `service_type_confidence`, `organization_service_types`, and `service_type_conflict` explain the decision.

For high accuracy with limited manual review, prioritize rows in this order: `unassigned` or `low` confidence first, then `service_type_conflict`, then description-based `medium` confidence. Guide-subcategory and guide-category mappings are the most reliable automated assignments; description keyword matches are deliberately conservative. No automated keyword system can guarantee correctness, so uncertain rows remain unassigned rather than being given a misleading category.


# Documentation of retained and added columns

## Original columns kept

These values come from the original resource file and are retained in the cleaned export, with clearer output names where noted:

- `guide_category` - original `Category`
- `guide_subcategory` - original `Subcategory`
- `address_line_2` - original `Address Line 2`
- `city` - original `City`
- `state` - original `State`
- `phone` - original `Phone Number`
- `description` - original `Description`
- `spanish_speaking` - original `Spanish-speaking (Y/N)`
- `pre_release` - original `Works w/ Currently Incarcerated/Pre-release (Y/N)`
- `ati_program` - original `ATI Program (Y/N)`
- `know_your_rights` - original `Know Your Rights Content (Y/N)`

## Added or enriched columns

These columns are created, renamed, normalized, derived, or added to support auditing and review:

- `resource_id` - generated numeric identifier such as `1`, `2`, or `3`.
- `org_name` - normalized organization name: only names written entirely in uppercase are converted to title case, while recognized acronyms such as `NYC`, `HIV`, `AIDS`, `CUNY`, and `GED` remain uppercase.
- `service_type` - standardized category from guide mappings or one unambiguous description match.
- `service_type_suggested` - possible categories for rows that remain uncertain.
- `street` - combined street number and street name.
- `zip` - cleaned and zero-padded ZIP code.
- `borough` - NYC borough derived from ZIP code.
- `latitude` and `longitude` - reserved for future geocoding and intentionally blank.
- `website` - normalized website URL.
- `row_type` - operational classification explained below.
- `status` and `source` - record status and provenance.
- `in_nomatch_file` - indicates a matching record in the NoMatch file; it does not import coordinates (can be deleted later).
- `service_type_source`, `service_type_confidence`, `organization_service_types`, and `service_type_conflict` - audit fields for classification decisions.
- `needs_review` and `review_notes` - identify rows requiring human review.

## How normalization is done

Normalization for consistent values:

- **Whitespace:** all CSV text values are converted to single spaces between words and trimmed at the beginning and end.
- **Organization names:** only names that are entirely uppercase are converted to title case. Known acronyms remain uppercase, and small connecting words such as `and`, `of`, and `for` are kept lowercase when they are not the first word. Names that were not entirely uppercase are not forcibly changed.
- **Street:** `Street Number` and `Street` are combined into the cleaned `street` field. Empty combined values become blank.
- **ZIP codes:** ZIP values are treated as text and padded with leading zeros to five characters. The known Brooklyn value `12116` is corrected to `11216` and recorded in the source note.
- **Websites:** a missing URL scheme is normalized by adding `https://`; existing URLs are preserved.
- **Service matching:** guide `Category` and `Subcategory` values are mapped to standardized service labels. Description matching is case-insensitive and uses regular expressions only when no guide match exists.
- **Organization matching:** organization names are converted to lowercase and all non-alphanumeric characters are removed to create `organization_key`, which is used only for comparing listings that appear to belong to the same organization.

## How `row_type` is created

`row_type` describes what kind of record a row represents. It is separate from `service_type`, which describes the resource subject area. The checks run in this order, and the first matching rule wins:

1. `mailing_address` - the street contains `box` or the source notes say `mailing address`.
2. `mapped` - a street address or latitude exists and the ZIP identifies a NYC borough.
3. `outside_nyc` - a street address or latitude exists, but no NYC borough is identified.
4. `hotline` - the phone or organization indicates a hotline, helpline, or lifeline; a standalone phone number also qualifies. A phone without a website is treated as a hotline.
5. `web_only` - a website exists but there is no usable physical address or hotline classification.
6. `cross_reference` - the description says to see the full listing elsewhere.
7. `unclassified` - none of the above evidence is available; the row is flagged for a row-type decision.

For example, a PO box with a website remains `mailing_address`, and a physical NYC address is classified as `mapped` before website checks are considered.


## How `service_type` is created

`service_type` is created through a conservative, two-stage classification process:

1. **Use the guide first.** The source `Category` and `Subcategory` values are normalized and matched to the standard service categories. When there is a reliable guide match, that category is used instead of inferring one from free text.
2. **Parse the description when no guide match exists.** The resource description is normalized for text matching, including case and common wording variations. Category-specific keywords and phrases are checked against the description.
3. **Assign only an unambiguous match.** If the description matches exactly one service category, that category is written to `service_type`.
4. **Keep ambiguous matches as suggestions.** If the description matches more than one category, the one or two strongest candidates are written to `service_type_suggested`; `service_type` remains blank and the row is marked for review.
5. **Leave unsupported rows blank.** If neither the guide fields nor the description provide enough evidence, no category is assigned. The reason is recorded in `review_notes` when applicable.

This approach avoids presenting an uncertain keyword match as a confirmed classification. The original guide fields and description remain in the export so that every assigned or suggested category can be reviewed.

### `service_type` values

- `Housing` - housing, shelters, homelessness, tenants, eviction, rent, or housing rights.
- `Food` - food pantries, meals, groceries, nutrition, or food assistance.
- `Health & Mental Health` - medical care, clinics, counseling, therapy, mental health, or general health services.
- `Addiction & Recovery` - substance-use services, addiction treatment, detox, recovery, or sobriety support.
- `Legal & Rights` - legal aid, attorneys, courts, civil rights, advocacy, or rights information.
- `Education` - schools, college, literacy, GED, ESOL, education, or training.
- `Employment` - jobs, employment, workforce programs, careers, vocational support, or resumes.
- `Financial Assistance` - benefits, taxes, public assistance, cash assistance, or financial counseling.
- `Family & Childcare` - childcare, day care, parenting, child welfare, or after-school support.
- `Reentry Support` - reentry after incarceration, parole, probation, or prison-release support.

The category is intentionally conservative: an unambiguous description match can populate `service_type`; multiple possible matches remain in `service_type_suggested` and are flagged for review.

## Exported column dictionary

The cleaned export contains one row per resource. Blank values mean that the source did not provide the information or that it still needs review.

| Column | Meaning |
| --- | --- |
| `resource_id` | Numeric identifier generated by row order, such as `1`, `2`, or `3`. |
| `org_name` | Organization name, with all-uppercase names normalized to title case while preserving known acronyms. |
| `service_type` | Main service category used for filtering. Existing guide mappings are applied first; if no mapping exists, the description is parsed. A description is assigned automatically only when it matches exactly one category. |
| `service_type_suggested` | One or two possible service categories for rows that remain unassigned. These are suggestions for human review, not confirmed classifications. |
| `guide_category` | Original `Category` value from the source guide. |
| `guide_subcategory` | Original `Subcategory` value from the source guide. |
| `street` | Combined street number and street name. PO boxes remain identifiable here and are classified as mailing addresses when detected. |
| `address_line_2` | Apartment, suite, floor, care-of, or other secondary address text. |
| `city` | Source city. |
| `state` | Source state abbreviation. |
| `zip` | ZIP code stored as text and padded to five characters where possible. |
| `borough` | NYC borough derived from ZIP code; blank for locations outside NYC or without a recognized NYC ZIP. |
| `latitude` | Reserved for future geocoding. Currently blank; no ArcGIS or NoMatch coordinates are imported. |
| `longitude` | Reserved for future geocoding. Currently blank; no ArcGIS or NoMatch coordinates are imported. |
| `phone` | Source phone value, preserved as text because it may include labels or multiple numbers. |
| `website` | Source website, with `https://` added when the source omitted a URL scheme. |
| `description` | Original resource description. This is one input used to suggest or assign `service_type`. |
| `spanish_speaking` | Original Spanish-speaking indicator (`Y`, `N`, or blank). |
| `pre_release` | Original indicator for working with currently incarcerated or pre-release people (`Y`, `N`, or blank). |
| `ati_program` | Original ATI Program indicator (`Y`, `N`, or blank). |
| `know_your_rights` | Original Know Your Rights Content indicator (`Y`, `N`, or blank). |
| `row_type` | Operational classification: `mapped`, `outside_nyc`, `mailing_address`, `hotline`, `web_only`, `cross_reference`, or `unclassified`. |
| `status` | Current record status. Existing source rows are set to `active`. |
| `source` | Provenance label for the primary resource dataset. |
| `in_nomatch_file` | Boolean indicating that a matching resource was also found in `NYPLConnections_NoMatch.csv`; this does not indicate that its coordinates were used. |
| `needs_review` | Boolean indicating that a person should check the row. |
| `review_notes` | Specific reasons for review, such as an unassigned service type, an ambiguous row type, a duplicate location, or an original source note. |

In [1]:
import os
import re

import pandas as pd

DATA_DIR = "../../data"
OUT_DIR = "../../data/cleaned"
os.makedirs(OUT_DIR, exist_ok=True)

In [2]:
# ---------- 0. LOAD (everything as text so ZIPs keep their zeros) ----------
def load(path):
    for enc in ("utf-8", "cp1252"):
        try:
            return pd.read_csv(path, dtype=str, encoding=enc, keep_default_na=False, na_values=[""])
        except UnicodeDecodeError:
            pass

df = load(os.path.join(DATA_DIR, "NYPL Connections Resources.csv"))
nm = load(os.path.join(DATA_DIR, "NYPLConnections_NoMatch.csv"))
df = df.apply(lambda c: c.str.replace(r"\s+", " ", regex=True).str.strip())   # trim spaces
df["note"] = ""

In [3]:
# ---------- 1. NAMELESS ROWS (address or hours text of the row above) ----------
for i in list(df.index[df["Organization Name"].isna()]):
    above = df.index[df.index.get_loc(i) - 1]
    if pd.notna(df.at[i, "Street"]):                       # second address -> copy details from above
        for col in df.columns:
            if col not in ["Street Number", "Address Line 2", "Street", "City", "State", "Zip Code", "note"]:
                df.at[i, col] = df.at[above, col]
        df.at[i, "note"] = "Address-only row, details copied from row above - verify"
    else:                                                  # text only -> add to note of row above
        df.at[above, "note"] += "Extra text: " + str(df.at[i, "Address Line 2"]) + " | "
        df = df.drop(index=i)
df = df.reset_index(drop=True)

In [4]:
# ---------- 2. NOMATCH PROVENANCE (matched by name+phone+website, in order) ----------
def key(d):
    k = d["Organization Name"].fillna("") + "|" + d["Phone Number"].fillna("") + "|" + d["Website"].fillna("")
    return k, k.groupby(k).cumcount()
df["k"], df["n"] = key(df)
nm["k"], nm["n"] = key(nm)
nm["in_nomatch_file"] = True
df = df.merge(nm[["k", "n", "in_nomatch_file"]], how="left", on=["k", "n"])
df["in_nomatch_file"] = df["in_nomatch_file"].fillna(False).astype(bool)
df["latitude"] = pd.NA
df["longitude"] = pd.NA

In [5]:
# ---------- 3. CLEAN ----------
ACRONYMS = ["NYC", "NYS", "NY", "HRA", "DSS", "DHS", "HIV", "AIDS", "LGBTQ", "LGBTQI", "CUNY",
            "SUNY", "YMCA", "YWCA", "GED", "HSE", "ESOL", "ATI", "PREA", "NEON", "II", "III"]
SMALL = ["and", "of", "for", "the", "in", "on", "to", "a", "an", "at", "with", "by", "or"]

def title(name):
    if re.sub(r"\([^)]*\)", "", name).replace(" ", "").isupper():          # only ALL CAPS names
        parts = re.split(r"(\([^)]*\))", name)                               # keep (BRACKETS) as they are
        words = lambda p: re.sub(r"[A-Za-z][A-Za-z'’]*", fix, p)
        def fix(m):
            w = m.group(0)
            if w in ACRONYMS: return w
            if w.lower() in SMALL and m.start() > 0: return w.lower()
            return w.capitalize()
        return "".join(p if p.startswith("(") else words(p) for p in parts)
    return name

df["org_name"] = df["Organization Name"].apply(title)
df["street"] = (df["Street Number"].fillna("") + " " + df["Street"].fillna("")).str.strip()
df["street"] = df["street"].replace("", None)
df["zip"] = df["Zip Code"].str.zfill(5)
df.loc[(df["City"] == "Brooklyn") & (df["zip"] == "12116"), "note"] += "ZIP 12116 changed to 11216 | "
df.loc[(df["City"] == "Brooklyn") & (df["zip"] == "12116"), "zip"] = "11216"
df["website"] = df["Website"].apply(lambda u: u if pd.isna(u) or u.startswith("http") else "https://" + u)

In [6]:
# ---------- 4. ENRICH ----------
def borough(z):
    if pd.isna(z): return None
    z = int(z)
    if 10001 <= z <= 10282: return "Manhattan"
    if 10301 <= z <= 10314: return "Staten Island"
    if 10451 <= z <= 10475: return "Bronx"
    if 11201 <= z <= 11256: return "Brooklyn"
    if 11004 <= z <= 11005 or 11101 <= z <= 11109 or 11351 <= z <= 11436 or 11691 <= z <= 11697: return "Queens"
df["borough"] = df["zip"].apply(borough)
df.loc[df["State"] != "NY", "borough"] = None

def row_type(r):
    phone = str(r["Phone Number"])
    if re.search(r"\bbox\b", str(r["Street"]), re.I) or "mailing address" in str(r["Notes"]).lower():
        return "mailing_address"
    if pd.notna(r["street"]) or pd.notna(r["latitude"]):
        return "mapped" if (pd.notna(r["borough"]) or pd.notna(r["latitude"])) else "outside_nyc"
    if re.search(r"hotline|helpline|lifeline", phone + str(r["org_name"]), re.I) or re.match(r"\d{3}(?![\d.])", phone):
        return "hotline"
    if pd.notna(r["website"]): return "web_only"
    if pd.notna(r["Phone Number"]): return "hotline"
    if "see the full listing" in str(r["Description"]).lower(): return "cross_reference"
    return "unclassified"
df["row_type"] = df.apply(row_type, axis=1)

by_sub = {"Organizations for People Without Homes": "Housing", "Tenant Rights & Housing Discrimination": "Housing",
          "Free & Low-Cost Medical Care": "Health & Mental Health", "HIV & AIDS": "Health & Mental Health",
          "General Health Information Resources": "Health & Mental Health", "Mental Health": "Health & Mental Health",
          "Helplines & Other Mental Health Resources": "Health & Mental Health",
          "Reproductive Health Information & Resources": "Health & Mental Health",
          "Treatment Programs & Organizations": "Addiction & Recovery", "Harm Reduction": "Addiction & Recovery",
          "Representation & Assistance": "Legal & Rights", "Grievances, Complaints & Government Agencies": "Legal & Rights",
          "Reporting Sexual Abuse & Assault (PREA)": "Legal & Rights", "Adult Literacy, HSE, & ESOL": "Education",
          "Continuing Education": "Education", "College Programs in Prison": "Education",
          "Computer Literacy & Digital Life Skills": "Education", "Food": "Food",
          "Financial Counseling & Rights": "Financial Assistance", "Taxes": "Financial Assistance"}
by_cat = {"Housing": "Housing", "Legal": "Legal & Rights", "Physical & Mental Health": "Health & Mental Health",
          "Addiction": "Addiction & Recovery", "Education": "Education", "Financial Assistance": "Financial Assistance",
          "Organizations for Job Seekers": "Employment", "Preparing for the Job Search": "Employment"}
df["service_type"] = df["Subcategory"].map(by_sub).fillna(df["Category"].map(by_cat))
df["service_type_source"] = df["service_type"].notna().map({True: "guide_mapping", False: "unassigned"})

# Use description text only for automatic classification when exactly one service matches.
hints = {"Food": r"\bfood\b|pantr|soup kitchen|\bmeals?\b|grocer|nutrition",
         "Family & Childcare": r"child ?care|day ?care|after-?school|parenting|child welfare",
         "Housing": r"housing|shelter|homeless|tenant|eviction|rent assistance",
         "Health & Mental Health": r"mental health|counsel|therap|medical|clinic|health care|\bhealthcare\b",
         "Addiction & Recovery": r"addiction|substance|recovery|detox|sobriety",
         "Legal & Rights": r"legal|attorney|lawyer|court|civil rights|advocacy",
         "Education": r"college|literacy|\bged\b|\besol\b|school|training|education",
         "Employment": r"\bjobs?\b|employ|workforce|career|vocational|resume",
         "Financial Assistance": r"financial|benefits|\btax|public assistance|cash assistance",
         "Reentry Support": r"re-?entry|formerly incarcerated|parole|probation|prison release"}

def matches(text):
    return [service for service, pattern in hints.items() if re.search(pattern, text, re.I)]

def description_match(description):
    found = matches(str(description))
    return found[0] if len(found) == 1 else None

missing = df["service_type"].isna()
df.loc[missing, "service_type"] = df.loc[missing, "Description"].apply(description_match)
df.loc[missing & df["service_type"].notna(), "service_type_source"] = "description_match"
df["service_type_confidence"] = df["service_type_source"].map({
    "guide_mapping": "high",
    "description_match": "medium",
    "unassigned": "unassigned",
})

def suggest(r):
    if pd.notna(r["service_type"]):
        return None
    text = " ".join(str(r[col]) for col in ["Description", "Subcategory", "Category"])
    found = matches(text)
    return "; ".join(found[:2]) or None

df["service_type_suggested"] = df.apply(suggest, axis=1)

# Apply an organization-wide type only when every known listing agrees.
def org_key(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())

df["organization_key"] = df["org_name"].apply(org_key)
org_types = df.groupby("organization_key")["service_type"].transform(
    lambda values: "; ".join(sorted(values.dropna().unique()))
)
df["organization_service_types"] = org_types.replace("", None)
df["service_type_conflict"] = df.groupby("organization_key")["service_type"].transform(
    lambda values: values.dropna().nunique() > 1
)

org_type_counts = df.groupby("organization_key")["service_type"].transform(
    lambda values: values.dropna().nunique()
)
org_consensus = org_types.where(org_type_counts == 1)
consensus_rows = df["service_type"].isna() & org_consensus.notna()
df.loc[consensus_rows, "service_type"] = org_consensus[consensus_rows]
df.loc[consensus_rows, "service_type_source"] = "organization_consensus"
df.loc[consensus_rows, "service_type_confidence"] = "medium"

# Conflicting organization listings are deliberately left blank rather than guessed.
conflict_rows = df["service_type_conflict"]
df.loc[conflict_rows, "service_type"] = pd.NA
df.loc[conflict_rows, "service_type_source"] = "organization_conflict"
df.loc[conflict_rows, "service_type_confidence"] = "unassigned"
df["service_type_suggested"] = df.apply(suggest, axis=1)

df["status"] = "active"
df["source"] = "NYPL Connections 2026 team sheet"
df["resource_id"] = list(range(1, len(df) + 1))

In [7]:
# ---------- 5. REVIEW FLAGS ----------
dup = df["org_name"] + "|" + df["street"].fillna("") + "|" + df["zip"].fillna("")
df.loc[df["street"].notna() & dup.duplicated(keep=False), "note"] += "Same place under several chapters | "
df.loc[df["in_nomatch_file"], "note"] += "In NoMatch file | "
df.loc[df["Notes"].notna(), "note"] += "Original note: " + df["Notes"].fillna("") + " | "
df.loc[df["service_type"].isna(), "note"] += "service_type to assign | "
df.loc[df["row_type"] == "unclassified", "note"] += "row_type needs a decision | "
df["review_notes"] = df["note"].str.strip(" |").replace("", None)
df["needs_review"] = df["review_notes"].notna()

In [8]:
# ---------- 6. SAVE ----------
rename = {"City": "city", "State": "state", "Phone Number": "phone", "Description": "description",
          "Category": "guide_category", "Subcategory": "guide_subcategory", "Address Line 2": "address_line_2",
          "Spanish-speaking (Y/N)": "spanish_speaking", "Works w/ Currently Incarcerated/Pre-release (Y/N)": "pre_release",
          "ATI Program (Y/N)": "ati_program", "Know Your Rights Content (Y/N)": "know_your_rights"}
out = df.rename(columns=rename)
cols = ["resource_id", "org_name", "service_type", "service_type_suggested", "guide_category", "guide_subcategory",
        "street", "address_line_2", "city", "state", "zip", "borough", "latitude", "longitude", "phone", "website",
        "description", "spanish_speaking", "pre_release", "ati_program", "know_your_rights",
        "row_type", "status", "source", "in_nomatch_file", "service_type_source", "service_type_confidence",
        "organization_service_types", "service_type_conflict", "needs_review", "review_notes"]
out[cols].to_csv(os.path.join(OUT_DIR, "resources_clean.csv"), index=False)
out[out["needs_review"]][cols].to_csv(os.path.join(OUT_DIR, "review_list.csv"), index=False)
print(len(out), "rows")
print(out["row_type"].value_counts())
print("service_type blank:", out["service_type"].isna().sum(), "| needs_review:", out["needs_review"].sum())

759 rows
row_type
mapped             395
web_only           169
hotline             71
cross_reference     50
outside_nyc         43
mailing_address     31
Name: count, dtype: int64
service_type blank: 244 | needs_review: 287
